<a href="https://colab.research.google.com/github/jromeo3415/BERT-adapt-to-NER/blob/main/ADL_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Chosen Project**: Named Entity Recognition (NER)

Loading conll2003 dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003", revision="refs/convert/parquet")
print(dataset)

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


Converting to dataframes and exploring dataset

In [ ]:
import pandas as pd

#converting to dataframe
train_df = dataset['train'].to_pandas()
validation_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

#defining ner tags dict
id_to_label = {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC', 7: 'B-MISC', 8: 'I-MISC'}

#printing without row or column truncation
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
  print(f"Train columns: {train_df.columns}")
  print(f"Number of training rows: {train_df['id'].count()}")

  print("\nFirst sentence's contents and corresponding NER tags:")
  print(train_df['tokens'][0])
  for tag in train_df['ner_tags'][0]:
    print(id_to_label[tag], end=", ")

Train columns: Index(['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'], dtype='object')
Number of training rows: 14041

First sentence's contents and corresponding NER tags:
['EU' 'rejects' 'German' 'call' 'to' 'boycott' 'British' 'lamb' '.']
B-ORG, O, B-MISC, O, O, O, B-MISC, O, O, 

**Defining BERT tokenizer and label alignment**

In [ ]:
# applys BERT word level tokenization to each sentence and returns tokens
def tokenize(tokens):

  #applying tokenizer
  enc = tokenizer(
      tokens["tokens"],
      is_split_into_words=True,
      return_offsets_mapping=True,
      truncation=True
  )

  aligned_labels = []

  for i in range(len(tokens["tokens"])):
    label_id = []
    previous_id = None
    word_ids = enc.word_ids(batch_index=i)

    for word_id in word_ids:
      if word_id == None:
        label_id.append(-100)
      elif word_id != previous_id:
        label_id.append(tokens["ner_tags"][i][word_id])
      else:
        label_id.append(-100)

      previous_id = word_id

    aligned_labels.append(label_id)

  enc["labels"] = aligned_labels
  return enc

**Applying tokenizaiton then label alignment for entire dataset**

In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
dataset = dataset.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

**Creating Trainer**

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ner_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


**Training model**

In [ ]:
from transformers import BertForTokenClassification, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id_to_label)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss
1,No log,0.049168
2,No log,0.040527
3,0.118343,0.039055


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=660, training_loss=0.09603301973053903, metrics={'train_runtime': 543.3738, 'train_samples_per_second': 77.521, 'train_steps_per_second': 1.215, 'total_flos': 1274087571937938.0, 'train_loss': 0.09603301973053903, 'epoch': 3.0})

**Evaluation**

In [ ]:
trainer.evaluate()

sentence = "The Google executives are hiring 2000 new employees in Lakeland, Florida."

inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)